In [61]:
import os
import glob
import polars as pl
from datetime import datetime

# --- 1. SET UP PATHS ---
first_glob = os.path.expanduser("~").replace("\\", "/")
test_path = f"{first_glob}/Concentrix Corporation"
if not os.path.exists(test_path):
    raise FileNotFoundError(f"Not found the path: {test_path}")

folder_paths = {
    "rta_intervals_output": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_ACTIVITY_INTERVALS',
    "iex_intervals_output": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_INTERVALS',
    "output_non_compliance": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_NON_COMPLIANCE',
    "hc_extend_by_month": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month',
    
    "req_path": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OU.xlsx',
    "prod_path": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_PRODUCTIVE_REVENUE',
    "wfm_path": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_WORKFORCE_VENDOR'
}

print("--- FULL FOLDER PATHS LIST ---")
for key, path in folder_paths.items():
    print(f"{key}: {path}")

print("-" * 60)

output_folder = folder_paths["output_non_compliance"]
os.makedirs(output_folder, exist_ok=True)

# --- 2. DEFINE DATA LOADER ---
def input_data(folder_path, sheet_name=None):
    """Loads all CSV and XLSX files from a folder, excluding specific years, and concatenates them."""
    file_paths = glob.glob(f"{folder_path}/*.xlsx") + glob.glob(f"{folder_path}/*.csv")
    df_list = []
    
    # Exclude files containing historical year strings
    exclude_strs = ["2023", "2024", "2025", "_23", "_24", "_25", " 23", " 24", " 25", "-23", "-24", "-25"]
    
    for file in file_paths:
        if any(old_year in file for old_year in exclude_strs):
            continue
            
        if file.endswith('.xlsx'):
            df = pl.read_excel(file, sheet_name=sheet_name)
        elif file.endswith('.csv'):
            try:
                df = pl.read_csv(file, encoding="utf-8")
            except Exception: # Fallback encoding if utf-8 fails
                df = pl.read_csv(file, encoding="ISO-8859-1", ignore_errors=True)
                
        # Cast all columns to string initially to avoid schema mismatch during concat
        df = df.with_columns(pl.all().cast(pl.String))
        df_list.append(df)
        
    return pl.concat(df_list, how='vertical') if df_list else pl.DataFrame()

# --- 3. LOAD & CLEAN DATA ---
# Load IEX Intervals
IEX_Intervals_Input = input_data(folder_paths["iex_intervals_output"]).with_columns([
    pl.col("OracleID").cast(pl.Int64), pl.col("IEX ID").cast(pl.Int64),
    pl.col("Week_Monday").str.to_date("%Y-%m-%d"),
    pl.col("Date_Converted").str.to_date("%Y-%m-%d"),
    pl.col("VNT_Intervals").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col("PST_Intervals").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col("Datetime_Start_Time").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col("Datetime_End_Time").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col("Duration").cast(pl.Float64)
]).filter(pl.col("Date_Converted").dt.year() == 2026)

# Load RTA Agent Activity
Agent_Activity_Intervals_Input = input_data(folder_paths["rta_intervals_output"]).with_columns([
    pl.col("OracleID").cast(pl.Int64), pl.col("IEX ID").cast(pl.Int64),
    pl.col("Week_Monday").str.to_date("%Y-%m-%d"),
    pl.col("Date_Converted").str.to_date("%Y-%m-%d"),
    pl.col("VNT_Intervals").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col("PST_Intervals").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col("Datetime_Start_Time").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col("Datetime_End_Time").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col("Duration").cast(pl.Float64),
    (pl.col("Target").cast(pl.Float64) / 3600).alias("Target")
]).filter(pl.col("Date_Converted").dt.year() == 2026)

# Load Headcount Data
HC_EXTEND_COMBINED = input_data(folder_paths["hc_extend_by_month"]).filter(pl.col("Year") == "2026").select([
    pl.col("Date").str.to_date("%Y-%m-%d"),
    (pl.col("Month") + "-01").str.to_date("%b-%y-%d").dt.strftime("%y_%m").alias("Month"),
    pl.col("Email Id"), pl.col("Alias"), pl.col("Designation"), 
    pl.col("Supervisor Name"), pl.col("LOB"), pl.col("Active")
])

# Extract Scheduled Activities boundaries from IEX
IEX_Scheduled_Activities = IEX_Intervals_Input.group_by(
    ["Week_Monday", "Date_Converted", "OracleID", "IEX ID", "Employee Name", "Email Id", 'VNT_Intervals', 'PST_Intervals', 'VNT_Interval_Range', 'PST_Interval_Range']
).agg([
    pl.col("Datetime_Start_Time").filter(pl.col("Scheduled Activity").str.contains("Break")).min().alias("Scheduled_Break_Start"),
    pl.col("Datetime_Start_Time").filter(pl.col("Scheduled Activity").str.contains("Lunch")).min().alias("Scheduled_Lunch_Start"),
    pl.col("Datetime_Start_Time").filter(pl.col("Scheduled Activity").str.contains("Training|Coaching")).min().alias("Scheduled_Training_Coaching_Start"),
    pl.col("Datetime_End_Time").max().alias("Scheduled_End_Shift")
])

# Create Base Merged Data
Base_Merged_Data = IEX_Scheduled_Activities.join(
    Agent_Activity_Intervals_Input,
    on=["Week_Monday", "Date_Converted", "OracleID", "IEX ID", "Employee Name", "Email Id", "VNT_Intervals", "PST_Intervals", "VNT_Interval_Range", "PST_Interval_Range"], 
    how="left"
).join(
    HC_EXTEND_COMBINED, 
    left_on=["Date_Converted", "Email Id"], 
    right_on=["Date", "Email Id"], 
    how="left"
).drop(["Month", "LOB"]).rename({"Month_right": "Month", "LOB_right": "LOB"})

Base_Merged_Data

--- FULL FOLDER PATHS LIST ---
rta_intervals_output: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_ACTIVITY_INTERVALS
iex_intervals_output: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_INTERVALS
output_non_compliance: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_NON_COMPLIANCE
hc_extend_by_month: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month
req_path: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OU.xlsx
prod_path: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_PRODUCTIVE_REVENUE
wfm_path: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_WORKFORCE_VENDOR
------------------------------------------------------------


Week_Monday,Date_Converted,OracleID,IEX ID,Employee Name,Email Id,VNT_Intervals,PST_Intervals,VNT_Interval_Range,PST_Interval_Range,Scheduled_Break_Start,Scheduled_Lunch_Start,Scheduled_Training_Coaching_Start,Scheduled_End_Shift,Agent Queue Group Name,Productive Aux Flag (Yes / No),Agent State,Datetime_Start_Time,Datetime_End_Time,Target,Duration,Month,Alias,Designation,Supervisor Name,LOB,Active
date,date,i64,i64,str,str,datetime[μs],datetime[μs],str,str,datetime[μs],datetime[μs],datetime[μs],datetime[μs],str,str,str,datetime[μs],datetime[μs],f64,f64,str,str,str,str,str,str
2026-04-06,2026-04-10,103293751,3109394,"""BUI NGOC THUAN VY""","""ngocthuanvy.bui@concentrix.com""",2026-04-10 23:30:00,2026-04-10 09:30:00,"""23:30-00:00""","""09:30-10:00""",null,2026-04-10 23:30:00,null,2026-04-11 00:00:00,"""Chat_OD_EN_Car_Activity""","""Yes""","""Lunch""",2026-04-10 23:30:00,2026-04-11 00:00:00,7.5,0.5,"""26_04""","""Mimi""","""Advisor I, Customer Service""","""Truong Thien Thanh Toan""","""Lodging""","""Yes"""
2026-01-05,2026-01-07,103171103,3096798,"""TRAN THI BICH LOI""","""thibichloi.tran@concentrix.com""",2026-01-07 11:00:00,2026-01-06 20:00:00,"""11:00-11:30""","""20:00-20:30""",null,2026-01-07 11:15:00,null,2026-01-07 11:30:00,"""Chat_OD_EN_Car_Activity""","""Yes""","""Available Chat""",2026-01-07 11:13:51,2026-01-07 11:30:00,9.566667,0.269167,"""26_01""","""Jadeli""","""Advisor I, Customer Service""","""Tran Sharon""","""Lodging""","""Yes"""
2026-01-05,2026-01-07,103171103,3096798,"""TRAN THI BICH LOI""","""thibichloi.tran@concentrix.com""",2026-01-07 11:00:00,2026-01-06 20:00:00,"""11:00-11:30""","""20:00-20:30""",null,2026-01-07 11:15:00,null,2026-01-07 11:30:00,"""Chat_OD_EN_Car_Activity""","""Yes""","""Available Chat""",2026-01-07 11:04:27,2026-01-07 11:13:51,9.566667,0.156667,"""26_01""","""Jadeli""","""Advisor I, Customer Service""","""Tran Sharon""","""Lodging""","""Yes"""
2026-01-05,2026-01-07,103171103,3096798,"""TRAN THI BICH LOI""","""thibichloi.tran@concentrix.com""",2026-01-07 11:00:00,2026-01-06 20:00:00,"""11:00-11:30""","""20:00-20:30""",null,2026-01-07 11:15:00,null,2026-01-07 11:30:00,"""Chat_OD_EN_Car_Activity""","""Yes""","""Available Chat""",2026-01-07 11:00:00,2026-01-07 11:04:27,9.566667,0.074167,"""26_01""","""Jadeli""","""Advisor I, Customer Service""","""Tran Sharon""","""Lodging""","""Yes"""
2026-01-12,2026-01-14,103169795,3096637,"""HUYNH QUANG ANH""","""quanganh.huynh@concentrix.com""",2026-01-15 02:30:00,2026-01-14 11:30:00,"""02:30-03:00""","""11:30-12:00""",null,null,null,2026-01-15 03:00:00,null,null,null,null,null,null,null,"""26_01""","""Jason""","""Advisor I, Customer Service""","""Tran Sharon""","""Lodging""","""Yes"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-04-13,2026-04-15,103060430,3052187,"""TO NGUYEN HOANG LONG""","""nguyenhoanglong.to@concentrix.…",2026-04-15 20:30:00,2026-04-15 06:30:00,"""20:30-21:00""","""06:30-07:00""",null,null,null,2026-04-15 21:00:00,"""Chat_OD_EN_Car_Activity""","""Yes""","""Available Chat""",2026-04-15 20:30:00,2026-04-15 20:35:03,7.5,0.084167,"""26_04""","""Drake""","""Advisor I, Customer Service""","""Mia Minh Lê""","""Lodging""","""Yes"""
2026-04-13,2026-04-15,103060430,3052187,"""TO NGUYEN HOANG LONG""","""nguyenhoanglong.to@concentrix.…",2026-04-15 20:30:00,2026-04-15 06:30:00,"""20:30-21:00""","""06:30-07:00""",null,null,null,2026-04-15 21:00:00,"""Chat_OD_EN_Car_Activity""","""Yes""","""Available Chat""",2026-04-15 20:49:56,2026-04-15 20:50:10,7.5,0.003889,"""26_04""","""Drake""","""Advisor I, Customer Service""","""Mia Minh Lê""","""Lodging""","""Yes"""
2026-04-13,2026-04-15,103060430,3052187,"""TO NGUYEN HOANG LONG""","""nguyenhoanglong.to@concentrix.…",2026-04-15 20:30:00,2026-04-15 06:30:00,"""20:30-21:00""","""06:30-07:00""",null,null,null,2026-04-15 21:00:00,"""Chat_OD_EN_Car_Activity""","""Yes""","""Available Chat""",2026-04-15 20:35:03,2026-04-15 20:42:56,7.5,0.131389,"""26_04""","""Drake""","""Advisor I, Customer Service""

In [62]:
# --- AVOIDANCE CALCULATION ---
avoiding_states = ["Break", "Lunch", "Training", "Coaching", 'Team Meeting', 'End of Shift']

# 1. Sort and calculate daily boundaries & total productive minutes
df_sorted = Base_Merged_Data.sort(["Email Id", "Date_Converted", "Datetime_Start_Time"]).with_columns([
    pl.col("Scheduled_End_Shift").max().over(["Email Id", "Date_Converted"]).alias("Daily_End_Shift"),
    (
        pl.col("Duration")
        .filter(pl.col("Productive Aux Flag (Yes / No)") == "Yes")
        .sum()
        .over(["Email Id", "Date_Converted"]) * 60
    ).round(2).alias("Total_Available_Mins")
])

# 2. Flag Exemptions & Violations
df_flagged = df_sorted.with_columns([
    # Check if the state matches the scheduled time
    (
        ((pl.col("Agent State") == "Break") & pl.col("Scheduled_Break_Start").is_not_null()) |
        ((pl.col("Agent State") == "Lunch") & pl.col("Scheduled_Lunch_Start").is_not_null()) |
        (pl.col("Agent State").is_in(["Training", "Coaching", "Team Meeting"]) & pl.col("Scheduled_Training_Coaching_Start").is_not_null()) |
        ((pl.col("Agent State") == "End of Shift") & pl.col("Scheduled_End_Shift").is_not_null()) |
        ((pl.col("Agent State") == "End of Shift") & pl.col("Daily_End_Shift").is_not_null() & (pl.col("Datetime_Start_Time") >= pl.col("Daily_End_Shift")))
    ).alias("Is_Scheduled_Exempt"),
    
    # Start Violation: Unproductive state followed by Productive state
    (
        (pl.col("Agent State").is_in(avoiding_states)) & (pl.col("Productive Aux Flag (Yes / No)") == "Yes") &
        (pl.col("Agent State").shift(-1).over(["Email Id", "Date_Converted"]).is_in(["Available Chat", "Outbound Call"])) &
        (pl.col("Productive Aux Flag (Yes / No)").shift(-1).over(["Email Id", "Date_Converted"]) == "Yes")
    ).alias("Is_Start_Violation"),
    
    # End Violation: Productive state preceded by Unproductive state
    (
        (pl.col("Agent State").is_in(["Available Chat", "Outbound Call"])) & (pl.col("Productive Aux Flag (Yes / No)") == "Yes") &
        (pl.col("Agent State").shift(1).over(["Email Id", "Date_Converted"]).is_in(avoiding_states)) &
        (pl.col("Productive Aux Flag (Yes / No)").shift(1).over(["Email Id", "Date_Converted"]) == "Yes")
    ).alias("Is_End_Violation")
]).with_columns(
    # Create violation groups for aggregating durations
    pl.col("Is_Start_Violation").cast(pl.Int32).cum_sum().over(["Email Id", "Date_Converted"]).alias("Violation_Group")
)

# 3. Aggregate Violations and finalize dataset
Agent_Avoidance_RawData = df_flagged.filter(pl.col("Is_Start_Violation") | pl.col("Is_End_Violation")).with_columns([
    pl.col("Datetime_Start_Time").min().over(["Email Id", "Date_Converted", "Violation_Group"]).alias("Event_Start_Time"),
    pl.col("Datetime_End_Time").max().over(["Email Id", "Date_Converted", "Violation_Group"]).alias("Event_End_Time")
]).with_columns([
    (pl.col("Event_End_Time") - pl.col("Event_Start_Time")).dt.total_seconds().alias("Duration_Secs")
]).with_columns([
    pl.col("Is_Scheduled_Exempt").any().over(["Email Id", "Date_Converted", "Violation_Group"]).alias("Group_Is_Scheduled"),
    (pl.col("Is_Start_Violation") & (pl.col("Duration_Secs") < 60)).any().over(["Email Id", "Date_Converted", "Violation_Group"]).alias("Group_Is_Short")
]).with_columns([
    # Assign exemption reason
    pl.when(pl.col("Group_Is_Scheduled") & pl.col("Group_Is_Short")).then(pl.lit("Scheduled activity & Duration < 60s"))
    .when(pl.col("Group_Is_Scheduled")).then(pl.lit("Scheduled activity"))
    .when(pl.col("Group_Is_Short")).then(pl.lit("Duration < 60s"))
    .otherwise(pl.lit(None)).alias("Exemption_Reason")
]).with_columns([
    # Final Flags and Formatting
    pl.when(pl.col("Exemption_Reason").is_not_null()).then(pl.lit("No")).otherwise(pl.lit("Yes")).alias("Non_Compliance"),
    pl.concat_str([
        pl.col("Date_Converted").cast(pl.String), pl.lit("_"), pl.col("Email Id"), pl.lit("_"),
        pl.col("Event_Start_Time").dt.strftime("%H%M%S"), pl.lit("-"), pl.col("Event_End_Time").dt.strftime("%H%M%S")
    ]).alias("Non_Compliance_Code"),
    pl.when(pl.col("Agent State").is_in(["Break", "Lunch", "Training", "Coaching", "Team Meeting"]))
    .then((pl.col("Duration_Secs") / 60.0).round(2))
    .otherwise(None).alias("Duration_Mins") 
]).select([
    "Month", "Date_Converted", "Email Id", "Employee Name", "LOB", "Alias", "Designation", "Supervisor Name", "Active",
    "VNT_Intervals", "Datetime_Start_Time", "Datetime_End_Time", 
    "Duration_Mins", "Total_Available_Mins", "Agent State", "Productive Aux Flag (Yes / No)", 
    "Scheduled_Break_Start", "Scheduled_Lunch_Start", "Scheduled_Training_Coaching_Start", "Scheduled_End_Shift", 
    "Non_Compliance", "Exemption_Reason", "Non_Compliance_Code"
]).rename({
    "Date_Converted": "Date",
    "Datetime_Start_Time": "State_Start_Time",
    "Datetime_End_Time": "State_End_Time",
    "Duration_Mins": "Avoidance_Duration_Mins", 
    "Productive Aux Flag (Yes / No)": "Is_Productive_Aux",
    "Non_Compliance": "Is_Avoidance_Violation"
})

In [63]:
# --- SCHEDULE ADHERENCE GAP ---
groupby_cols = ["Month", "Week_Monday", "Date_Converted", "Alias", "LOB", "Designation", "Supervisor Name", "Active", "Employee Name", "Email Id", "OracleID", "IEX ID", "VNT_Intervals", "PST_Intervals", "VNT_Interval_Range", "PST_Interval_Range"]

# 1. Aggregate Actual RTA Data
Agent_Activity_Adherence = Base_Merged_Data.group_by(groupby_cols).agg([
    # Break
    pl.col("Datetime_Start_Time").filter(pl.col("Agent State").str.contains("Break") & (pl.col("Productive Aux Flag (Yes / No)") == "No")).min().alias("Actual_Break_Start"),
    pl.col("Datetime_End_Time").filter(pl.col("Agent State").str.contains("Break") & (pl.col("Productive Aux Flag (Yes / No)") == "No")).max().alias("Actual_Break_End"),
    (pl.col("Duration").filter(pl.col("Agent State").str.contains("Break") & (pl.col("Productive Aux Flag (Yes / No)") == "No")).sum() * 60).round(2).alias("Actual_Break_Duration"),
    
    # Lunch
    pl.col("Datetime_Start_Time").filter(pl.col("Agent State").str.contains("Lunch") & (pl.col("Productive Aux Flag (Yes / No)") == "No")).min().alias("Actual_Lunch_Start"),
    pl.col("Datetime_End_Time").filter(pl.col("Agent State").str.contains("Lunch") & (pl.col("Productive Aux Flag (Yes / No)") == "No")).max().alias("Actual_Lunch_End"),
    (pl.col("Duration").filter(pl.col("Agent State").str.contains("Lunch") & (pl.col("Productive Aux Flag (Yes / No)") == "No")).sum() * 60).round(2).alias("Actual_Lunch_Duration"),
    
    # Training / Coaching
    pl.col("Datetime_Start_Time").filter(pl.col("Agent State").is_in(['Team Meeting', 'Coaching', 'Training']) & (pl.col("Productive Aux Flag (Yes / No)") == "No")).min().alias("Actual_TC_Start"),
    pl.col("Datetime_End_Time").filter(pl.col("Agent State").is_in(['Team Meeting', 'Coaching', 'Training']) & (pl.col("Productive Aux Flag (Yes / No)") == "No")).max().alias("Actual_TC_End"),
    (pl.col("Duration").filter(pl.col("Agent State").is_in(['Team Meeting', 'Coaching', 'Training']) & (pl.col("Productive Aux Flag (Yes / No)") == "No")).sum() * 60).round(2).alias("Actual_TC_Duration")
])

# 2. Aggregate Scheduled IEX Data
IEX_Adherence = IEX_Intervals_Input.group_by(["Date_Converted", "Email Id", "VNT_Intervals"]).agg([
    pl.col("Datetime_Start_Time").min().alias("Scheduled_Start"),
    pl.col("Datetime_End_Time").max().alias("Scheduled_End"),
    (pl.col("Duration").filter(pl.col("Scheduled Activity").str.contains("Break")).sum() * 60).round(2).alias("Scheduled_Break_Duration"),
    (pl.col("Duration").filter(pl.col("Scheduled Activity").str.contains("Lunch")).sum() * 60).round(2).alias("Scheduled_Lunch_Duration"),
    (pl.col("Duration").filter(pl.col("Scheduled Activity").str.contains("Training|Coaching")).sum() * 60).round(2).alias("Scheduled_TC_Duration")
])

# 3. Join and calculate Gaps (Actual - Scheduled -> Positive = Over, Negative = Under)
Merged_Adherence = Agent_Activity_Adherence.join(IEX_Adherence, on=["Date_Converted", "Email Id", "VNT_Intervals"], how="left").with_columns([
    (pl.col("Actual_Break_Duration").fill_null(0) - pl.col("Scheduled_Break_Duration").fill_null(0)).alias("Gap_Break"),
    (pl.col("Actual_Lunch_Duration").fill_null(0) - pl.col("Scheduled_Lunch_Duration").fill_null(0)).alias("Gap_Lunch"),
    (pl.col("Actual_TC_Duration").fill_null(0) - pl.col("Scheduled_TC_Duration").fill_null(0)).alias("Gap_TC")
])

# 4. Unpivot / Restructure data by Activity Type
df_break = Merged_Adherence.select(groupby_cols + [
    pl.lit("Break").alias("Activity_Type"), pl.col("Actual_Break_Start").alias("Actual_Start_Time"), pl.col("Actual_Break_End").alias("Actual_End_Time"),
    pl.col("Scheduled_Start"), pl.col("Scheduled_End"), pl.col("Scheduled_Break_Duration").alias("Scheduled_Duration"),
    pl.col("Actual_Break_Duration").alias("Actual_Duration"), pl.col("Gap_Break").alias("Gap")
])

df_lunch = Merged_Adherence.select(groupby_cols + [
    pl.lit("Lunch").alias("Activity_Type"), pl.col("Actual_Lunch_Start").alias("Actual_Start_Time"), pl.col("Actual_Lunch_End").alias("Actual_End_Time"),
    pl.col("Scheduled_Start"), pl.col("Scheduled_End"), pl.col("Scheduled_Lunch_Duration").alias("Scheduled_Duration"),
    pl.col("Actual_Lunch_Duration").alias("Actual_Duration"), pl.col("Gap_Lunch").alias("Gap")
])

df_tc = Merged_Adherence.select(groupby_cols + [
    pl.lit("Training | Coaching").alias("Activity_Type"), pl.col("Actual_TC_Start").alias("Actual_Start_Time"), pl.col("Actual_TC_End").alias("Actual_End_Time"),
    pl.col("Scheduled_Start"), pl.col("Scheduled_End"), pl.col("Scheduled_TC_Duration").alias("Scheduled_Duration"),
    pl.col("Actual_TC_Duration").alias("Actual_Duration"), pl.col("Gap_TC").alias("Gap")
])

# 5. Final Formatting and Tolerance Status assignment
Schedule_Adherence_Gap = pl.concat([df_break, df_lunch, df_tc]).filter(
    (pl.col("Scheduled_Duration").fill_null(0) > 0) | (pl.col("Actual_Duration").fill_null(0) > 0)
).sort(["Email Id", "Date_Converted", "VNT_Intervals", "Activity_Type"]).rename({
    "Date_Converted": "Date",
    "Actual_Start_Time": "Actual_Activity_Start",
    "Actual_End_Time": "Actual_Activity_End",
    "Scheduled_Start": "Scheduled_Activity_Start",
    "Scheduled_End": "Scheduled_Activity_End",
    "Actual_Duration": "Actual_Duration_Mins",
    "Scheduled_Duration": "Scheduled_Duration_Mins",
    "Gap": "Duration_Gap_Mins"
}).with_columns(
    # Set status using a +/- 2 minute tolerance for 'Matched'
    pl.when(pl.col("Duration_Gap_Mins") > 2).then(pl.lit("Over Scheduled"))
    .when(pl.col("Duration_Gap_Mins") < -2).then(pl.lit("Under Scheduled"))
    .otherwise(pl.lit("Matched"))
    .alias("Gap_Status")
)

In [64]:
# --- AGENT REST COMPLIANCE ---
actual_rest_raw = Base_Merged_Data.filter(
    pl.col("Agent State").is_in(["Break", "Lunch"])
).sort(["Email Id", "Date_Converted", "Datetime_Start_Time"])

# 1. Identify distinct Rest Sessions (state change OR gap > 60s)
actual_rest_raw = actual_rest_raw.with_columns(
    (
        (pl.col("Agent State") != pl.col("Agent State").shift(1).over(["Email Id", "Date_Converted"])) |
        ((pl.col("Datetime_Start_Time") - pl.col("Datetime_End_Time").shift(1).over(["Email Id", "Date_Converted"])).dt.total_seconds() > 60)
    ).fill_null(True).alias("Is_New_Session")
).with_columns(
    pl.col("Is_New_Session").cast(pl.Int32).cum_sum().over(["Email Id", "Date_Converted"]).alias("Session_ID")
)

# 2. Aggregate sessions and remove micro-sessions (< 1 min)
Agent_Rest_Compliance_Log = actual_rest_raw.group_by([
    "Month", "Date_Converted", "Email Id", "Employee Name", "Alias", "LOB", "Designation", "Supervisor Name", "Active", "Agent State", "Session_ID"
]).agg([
    pl.col("Datetime_Start_Time").min().alias("Session_Start_Time"),
    pl.col("Datetime_End_Time").max().alias("Session_End_Time"),
    (pl.col("Duration").filter(pl.col("Productive Aux Flag (Yes / No)") == "No").sum() * 60).round(2).alias("Total_Duration_Mins") 
]).sort(["Email Id", "Date_Converted", "Session_Start_Time"]).filter(
    pl.col("Total_Duration_Mins").fill_null(0) >= 1.0
)

# 3. Apply limit logic and bucket exceptions
Agent_Rest_Compliance_Log = Agent_Rest_Compliance_Log.with_columns(
    pl.col("Session_Start_Time").rank("ordinal").over(["Email Id", "Date_Converted", "Agent State"]).alias("Rank")
).with_columns(
    # Name the session dynamically (e.g., Break_1, Lunch_1)
    pl.when(pl.col("Agent State") == "Break").then(pl.lit("Break_") + pl.col("Rank").cast(pl.String))
    .when((pl.col("Agent State") == "Lunch") & (pl.col("Rank") == 1)).then(pl.lit("Lunch"))
    .otherwise(pl.lit("Lunch_") + pl.col("Rank").cast(pl.String)).alias("Detail_Rest_Type")
).with_columns([
    # Flag over rest (Break limit: 15m, Lunch limit: 60m)
    pl.when((pl.col("Agent State") == "Break") & (pl.col("Total_Duration_Mins") > 15.0)).then(pl.lit("Yes (>15m)"))
    .when((pl.col("Agent State") == "Lunch") & (pl.col("Total_Duration_Mins") > 60.0)).then(pl.lit("Yes (>60m)"))
    .otherwise(pl.lit("No")).alias("Is_Over_Rest"),
    
    # Calculate excess duration
    pl.when((pl.col("Agent State") == "Break") & (pl.col("Total_Duration_Mins") > 15.0)).then((pl.col("Total_Duration_Mins") - 15.0).round(2))
    .when((pl.col("Agent State") == "Lunch") & (pl.col("Total_Duration_Mins") > 60.0)).then((pl.col("Total_Duration_Mins") - 60.0).round(2))
    .otherwise(pl.lit(0.0)).alias("Over_Duration_Mins")
]).select([
    "Month", "Date_Converted", "Email Id", "Employee Name", "Alias", "LOB", "Designation", "Supervisor Name", "Active", 
    "Detail_Rest_Type", "Session_Start_Time", "Session_End_Time", "Total_Duration_Mins", 
    "Is_Over_Rest", "Over_Duration_Mins"
]).rename({
    "Date_Converted": "Date",
    "Detail_Rest_Type": "Rest_Session_Name",
    "Total_Duration_Mins": "Actual_Rest_Duration_Mins",
    "Is_Over_Rest": "Is_Exceeding_Limit",
    "Over_Duration_Mins": "Exceeded_By_Mins"
}).with_columns(
    # Bucket the exceeded time
    pl.when(pl.col("Exceeded_By_Mins") > 10).then(pl.lit("> 10 min"))
    .when(pl.col("Exceeded_By_Mins") > 5).then(pl.lit("5-10 min"))
    .when(pl.col("Exceeded_By_Mins") >= 1).then(pl.lit("1-5 min"))
    .when(pl.col("Exceeded_By_Mins") > 0).then(pl.lit("< 1 min"))
    .otherwise(pl.lit("Not Exceeded"))
    .alias("Exceed_Bucket")
)

In [65]:
# --- INTERVAL COMPLIANCE & MERGING ---
lob_mapping = {
    "GEN_GEN_EN_GCS_GLG_CHT": "Lodging chat",
    "GEN_GEN_EN_GCS_GNL_CHT": "Non-Lodging chat",
    "GEN_GEN_EN_GCS_GLG_CHT_Concentrix": "Lodging chat",
    "GEN_GEN_EN_GCS_GNL_CHT_Concentrix": "Non-Lodging chat"
}

vnt_tz_expr = (
    pl.col("PST_Intervals")
    .dt.replace_time_zone("America/Los_Angeles", non_existent="null", ambiguous="earliest")
    .dt.convert_time_zone("Asia/Ho_Chi_Minh")
    .dt.replace_time_zone(None)
    .alias("VNT_Intervals")
)

# 1. Process Required Heads (OU_Data)
OU_Data = pl.read_excel(folder_paths["req_path"], has_header=False)
new_headers = [str(val) for val in OU_Data.row(0)]
rename_mapping = dict(zip(OU_Data.columns, new_headers))

OU_Data = OU_Data.slice(1).rename(rename_mapping).unpivot(
    index=["LOB", "Site", "Interval"], variable_name="Date", value_name="Req"
).with_columns([
    pl.col("Date").str.slice(0, 10).str.to_date("%Y-%m-%d"),
    pl.col("Interval").str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False).dt.time(),
    pl.col("Req").cast(pl.Float64, strict=False)
]).with_columns(
    pl.col("Date").dt.combine(pl.col("Interval")).alias("PST_Intervals")
).with_columns(vnt_tz_expr)

# 2. Process Revenue / Productive Data
Productive_Revenue_Data = input_data(folder_paths["prod_path"]).with_columns([
    pl.col("Productive Hours (Sum)").cast(pl.Float64, strict=False),
    pl.col("Productive Idle Hour (Sum)").cast(pl.Float64, strict=False),
    pl.col("Handle Count (Sum)").cast(pl.Float64, strict=False)
]).group_by(["Business Location", "Interval", "Forecast Group Name"]).agg([
    pl.col("Productive Hours (Sum)").sum(),
    pl.col("Productive Idle Hour (Sum)").sum(),
    pl.col("Handle Count (Sum)").sum()
]).rename({
    "Business Location": "Site", "Forecast Group Name": "LOB", "Interval": "PST_Intervals"
}).with_columns([
    pl.col("LOB").replace_strict(lob_mapping, default=pl.col("LOB")),
    pl.col("PST_Intervals").str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
    pl.when(pl.col("Productive Hours (Sum)") == 0).then(None).otherwise(
        (pl.col("Productive Hours (Sum)") - pl.col("Productive Idle Hour (Sum)")) / pl.col("Productive Hours (Sum)")
    ).alias("Occupancy_Site_Level")
]).with_columns(vnt_tz_expr)

# 3. Process Workforce Vendor Data
vendor_cols = ["Forecast Productive Hour (Sum)", "Productive Hour (Sum)", "Staffing Attainment (Pct)", "Interval Compliance (Pct)", "Occupancy", "Billable Hours (Sum)"]

Workforce_Vendor_Data = input_data(folder_paths["wfm_path"]).with_columns(
    [pl.col(c).cast(pl.Float64, strict=False) for c in vendor_cols]
).group_by(["Vendor Forecast Group Name", "Interval Time"]).agg(
    [pl.col(c).sum() for c in vendor_cols]
).rename({
    "Vendor Forecast Group Name": "LOB", "Interval Time": "PST_Intervals"
}).with_columns([
    pl.col("LOB").replace_strict(lob_mapping, default=pl.col("LOB")),
    pl.col("PST_Intervals").str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
    pl.when(pl.col("Interval Compliance (Pct)") >= 1).then(pl.lit("Met")).otherwise(pl.lit("Missed")).alias("IC Status")
]).with_columns(vnt_tz_expr)

# 4. Final Data Merging with renamed productive columns
revenue_subset = Productive_Revenue_Data.select([
    "LOB", "PST_Intervals", "Site", 
    pl.col("Productive Hours (Sum)").alias("Site_Productive_Hours"), # Rename HCM Site productive
    "Handle Count (Sum)", "Occupancy_Site_Level"
])

ou_subset = OU_Data.select(["LOB", "Site", "PST_Intervals", "Req"])

Final_Merged_Data = Workforce_Vendor_Data.rename({
    "Productive Hour (Sum)": "Total_LOB_Productive_Hours" # Rename Global/Vendor productive
}).join(
    revenue_subset, on=["LOB", "PST_Intervals"], how="left", suffix="_Site_Level"
).join(
    ou_subset, on=["LOB", "Site", "PST_Intervals"], how="left"
).rename(
    {"Req": "Req_Site_Level"}
).with_columns([
    # Calculation using new names
    pl.when(pl.col("Req_Site_Level") == 0).then(None).otherwise(
        pl.col("Site_Productive_Hours") / pl.col("Req_Site_Level")
    ).alias("Staffing Attainment (Pct)_Site_Level"),
    
    (pl.col("Forecast Productive Hour (Sum)") * 2).alias("Forecast Heads"),
    (pl.col("Total_LOB_Productive_Hours") * 2).alias("Actual Heads"), # Renamed reference
    (pl.col("Site_Productive_Hours") * 2).alias("Site Actual Heads"), # Renamed reference
    (pl.col("Req_Site_Level") * 2).alias("Site Req Heads"),
    
    # Date/Time formatting
    pl.col("PST_Intervals").dt.strftime("%Y-%m").alias("PST_Month"),
    pl.col("PST_Intervals").dt.date().alias("PST_Date"),
    pl.concat_str([
        pl.col("PST_Intervals").dt.strftime("%H:%M"), pl.lit("-"),
        (pl.col("PST_Intervals") + pl.duration(minutes=30)).dt.strftime("%H:%M")
    ]).alias("PST_Interval_Range"),
    
    pl.col("VNT_Intervals").dt.date().alias("VNT_Date"),
    pl.concat_str([
        pl.col("VNT_Intervals").dt.strftime("%H:%M"), pl.lit("-"),
        (pl.col("VNT_Intervals") + pl.duration(minutes=30)).dt.strftime("%H:%M")
    ]).alias("VNT_Interval_Range")
]).select([
    "LOB", "PST_Month", "PST_Date", "PST_Intervals", "PST_Interval_Range",
    "VNT_Date", "VNT_Intervals", "VNT_Interval_Range",
    "Forecast Productive Hour (Sum)", "Forecast Heads", 
    "Total_LOB_Productive_Hours", "Actual Heads", # New Names
    "Staffing Attainment (Pct)", "Interval Compliance (Pct)", "IC Status", "Occupancy", "Billable Hours (Sum)",
    "Site", "Req_Site_Level", "Site Req Heads", 
    "Site_Productive_Hours", "Site Actual Heads", # New Names
    "Staffing Attainment (Pct)_Site_Level", "Occupancy_Site_Level", "Handle Count (Sum)"
])

In [66]:
Base_Merged_Data

Week_Monday,Date_Converted,OracleID,IEX ID,Employee Name,Email Id,VNT_Intervals,PST_Intervals,VNT_Interval_Range,PST_Interval_Range,Scheduled_Break_Start,Scheduled_Lunch_Start,Scheduled_Training_Coaching_Start,Scheduled_End_Shift,Agent Queue Group Name,Productive Aux Flag (Yes / No),Agent State,Datetime_Start_Time,Datetime_End_Time,Target,Duration,Month,Alias,Designation,Supervisor Name,LOB,Active
date,date,i64,i64,str,str,datetime[μs],datetime[μs],str,str,datetime[μs],datetime[μs],datetime[μs],datetime[μs],str,str,str,datetime[μs],datetime[μs],f64,f64,str,str,str,str,str,str
2026-04-06,2026-04-10,103293751,3109394,"""BUI NGOC THUAN VY""","""ngocthuanvy.bui@concentrix.com""",2026-04-10 23:30:00,2026-04-10 09:30:00,"""23:30-00:00""","""09:30-10:00""",null,2026-04-10 23:30:00,null,2026-04-11 00:00:00,"""Chat_OD_EN_Car_Activity""","""Yes""","""Lunch""",2026-04-10 23:30:00,2026-04-11 00:00:00,7.5,0.5,"""26_04""","""Mimi""","""Advisor I, Customer Service""","""Truong Thien Thanh Toan""","""Lodging""","""Yes"""
2026-01-05,2026-01-07,103171103,3096798,"""TRAN THI BICH LOI""","""thibichloi.tran@concentrix.com""",2026-01-07 11:00:00,2026-01-06 20:00:00,"""11:00-11:30""","""20:00-20:30""",null,2026-01-07 11:15:00,null,2026-01-07 11:30:00,"""Chat_OD_EN_Car_Activity""","""Yes""","""Available Chat""",2026-01-07 11:13:51,2026-01-07 11:30:00,9.566667,0.269167,"""26_01""","""Jadeli""","""Advisor I, Customer Service""","""Tran Sharon""","""Lodging""","""Yes"""
2026-01-05,2026-01-07,103171103,3096798,"""TRAN THI BICH LOI""","""thibichloi.tran@concentrix.com""",2026-01-07 11:00:00,2026-01-06 20:00:00,"""11:00-11:30""","""20:00-20:30""",null,2026-01-07 11:15:00,null,2026-01-07 11:30:00,"""Chat_OD_EN_Car_Activity""","""Yes""","""Available Chat""",2026-01-07 11:04:27,2026-01-07 11:13:51,9.566667,0.156667,"""26_01""","""Jadeli""","""Advisor I, Customer Service""","""Tran Sharon""","""Lodging""","""Yes"""
2026-01-05,2026-01-07,103171103,3096798,"""TRAN THI BICH LOI""","""thibichloi.tran@concentrix.com""",2026-01-07 11:00:00,2026-01-06 20:00:00,"""11:00-11:30""","""20:00-20:30""",null,2026-01-07 11:15:00,null,2026-01-07 11:30:00,"""Chat_OD_EN_Car_Activity""","""Yes""","""Available Chat""",2026-01-07 11:00:00,2026-01-07 11:04:27,9.566667,0.074167,"""26_01""","""Jadeli""","""Advisor I, Customer Service""","""Tran Sharon""","""Lodging""","""Yes"""
2026-01-12,2026-01-14,103169795,3096637,"""HUYNH QUANG ANH""","""quanganh.huynh@concentrix.com""",2026-01-15 02:30:00,2026-01-14 11:30:00,"""02:30-03:00""","""11:30-12:00""",null,null,null,2026-01-15 03:00:00,null,null,null,null,null,null,null,"""26_01""","""Jason""","""Advisor I, Customer Service""","""Tran Sharon""","""Lodging""","""Yes"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-04-13,2026-04-15,103060430,3052187,"""TO NGUYEN HOANG LONG""","""nguyenhoanglong.to@concentrix.…",2026-04-15 20:30:00,2026-04-15 06:30:00,"""20:30-21:00""","""06:30-07:00""",null,null,null,2026-04-15 21:00:00,"""Chat_OD_EN_Car_Activity""","""Yes""","""Available Chat""",2026-04-15 20:30:00,2026-04-15 20:35:03,7.5,0.084167,"""26_04""","""Drake""","""Advisor I, Customer Service""","""Mia Minh Lê""","""Lodging""","""Yes"""
2026-04-13,2026-04-15,103060430,3052187,"""TO NGUYEN HOANG LONG""","""nguyenhoanglong.to@concentrix.…",2026-04-15 20:30:00,2026-04-15 06:30:00,"""20:30-21:00""","""06:30-07:00""",null,null,null,2026-04-15 21:00:00,"""Chat_OD_EN_Car_Activity""","""Yes""","""Available Chat""",2026-04-15 20:49:56,2026-04-15 20:50:10,7.5,0.003889,"""26_04""","""Drake""","""Advisor I, Customer Service""","""Mia Minh Lê""","""Lodging""","""Yes"""
2026-04-13,2026-04-15,103060430,3052187,"""TO NGUYEN HOANG LONG""","""nguyenhoanglong.to@concentrix.…",2026-04-15 20:30:00,2026-04-15 06:30:00,"""20:30-21:00""","""06:30-07:00""",null,null,null,2026-04-15 21:00:00,"""Chat_OD_EN_Car_Activity""","""Yes""","""Available Chat""",2026-04-15 20:35:03,2026-04-15 20:42:56,7.5,0.131389,"""26_04""","""Drake""","""Advisor I, Customer Service""

In [71]:
# ==========================================
# --- FINAL AGGREGATION (IEX + RTA) ---
# ==========================================

# ------------------------------------------
# 1. IEX AGGREGATION (Scheduled Data - In Heads)
# ------------------------------------------
leave_list = ['Unscheduled', 'PTO', 'Termination', 'Offline', 'Paid Leave']

lob_mapping_extended = {
    "GEN_GEN_EN_GCS_GLG_CHT": "Lodging chat",
    "GEN_GEN_EN_GCS_GNL_CHT": "Non-Lodging chat",
    "GEN_GEN_EN_GCS_GLG_CHT_Concentrix": "Lodging chat",
    "GEN_GEN_EN_GCS_GNL_CHT_Concentrix": "Non-Lodging chat",
    "Lodging": "Lodging chat",
    "Lodging_Nesting": "Lodging chat",
    "Non_Lodging": "Non-Lodging chat"
}

hc_subset = HC_EXTEND_COMBINED.select(["Date", "Email Id", "LOB"])

IEX_Aggregated = IEX_Intervals_Input.join(
    hc_subset, left_on=["Date_Converted", "Email Id"], right_on=["Date", "Email Id"], how="left"
).with_columns(
    pl.col("LOB").replace_strict(lob_mapping_extended, default=pl.col("LOB"))
).group_by(
    ["LOB", "VNT_Intervals", "PST_Intervals"]
).agg([
    (pl.col("Duration").filter(pl.col("Scheduled Activity").is_in(["Open Time", "Extra Hours"])).sum() * 2).alias("Scheduled_Open_Time"),
    (pl.col("Duration").filter(pl.col("Scheduled Activity").str.contains("Break")).sum() * 2).alias("Scheduled_Break"),
    (pl.col("Duration").filter(pl.col("Scheduled Activity").str.contains("Lunch")).sum() * 2).alias("Scheduled_Lunch"),
    (pl.col("Duration").filter(pl.col("Scheduled Activity").str.contains("Training|Coaching")).sum() * 2).alias("Scheduled_Training/Coaching"),
    (pl.col("Duration").filter(pl.col("Scheduled Activity").is_in(leave_list)).sum() * 2).alias("Scheduled_Leave"),
    (pl.col("Duration").filter(pl.col("Scheduled Activity") == "No Call/No Show").sum() * 2).alias("Scheduled_NCNS")
])

# ------------------------------------------
# 2. RTA AGGREGATION (Actual Data - In Heads)
# ------------------------------------------
rta_lob_mapping = {
    "Chat_OD_EN_Lodging": "Lodging chat",
    "Chat_OD_EN_Car_Activity": "Lodging chat",
    "Chat_OD_Retail_Lodging_GLB_EN_Nesting": "Lodging chat",
    "Chat_OD_EN_Dual_GDS": "Non-Lodging chat",
    "Chat_OD_EN_Amadeus": "Non-Lodging chat"
}

tc_states = ["Training", "Coaching", "Team Meeting", "Restroom", "System Issue"]

RTA_AGGREGATION = Agent_Activity_Intervals_Input.filter(
).with_columns(
    pl.col("Agent Queue Group Name").replace_strict(rta_lob_mapping, default=None).alias("LOB")
).filter(
    pl.col("LOB").is_not_null()
).group_by(
    ["LOB", "VNT_Intervals"]
).agg([
    (pl.col("Duration").filter(
        pl.col("Agent State").str.contains("Break") & 
        (pl.col("Productive Aux Flag (Yes / No)") == "No")
    ).sum() * 2).alias("Actual_Break"),
    
    (pl.col("Duration").filter(
        pl.col("Agent State").str.contains("Lunch") & 
        (pl.col("Productive Aux Flag (Yes / No)") == "No")
    ).sum() * 2).alias("Actual_Lunch"),
    
    (pl.col("Duration").filter(
        pl.col("Agent State").is_in(tc_states) & 
        (pl.col("Productive Aux Flag (Yes / No)") == "No")
    ).sum() * 2).alias("Actual_Training_Coaching")
])

# ------------------------------------------
# 3. FINAL AGGREGATION (Merge IEX and RTA)
# ------------------------------------------
FINAL_AGGREGATION = IEX_Aggregated.join(
    RTA_AGGREGATION,
    on=["LOB", "VNT_Intervals"],
    how="left"
).fill_null(0)

FINAL_AGGREGATION

LOB,VNT_Intervals,PST_Intervals,Scheduled_Open_Time,Scheduled_Break,Scheduled_Lunch,Scheduled_Training/Coaching,Scheduled_Leave,Scheduled_NCNS,Actual_Break,Actual_Lunch,Actual_Training_Coaching
str,datetime[μs],datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Non-Lodging chat""",2026-03-17 12:00:00,2026-03-16 22:00:00,5.5,0.0,5.5,0.0,0.0,1.0,0.0,4.239444,0.0
"""Lodging chat""",2026-03-15 02:00:00,2026-03-14 12:00:00,12.166667,0.0,9.833333,0.0,3.0,2.0,0.0,9.52,0.0
"""Lodging chat""",2026-04-05 13:00:00,2026-04-04 23:00:00,12.166667,1.833333,2.0,0.0,5.0,3.0,0.988333,0.622778,0.0
"""Lodging chat""",2026-03-22 03:30:00,2026-03-21 13:30:00,23.333333,1.666667,1.0,0.0,1.0,0.0,0.877222,2.898333,0.0
"""Lodging chat""",2026-01-07 02:30:00,2026-01-06 11:30:00,22.5,0.0,5.5,0.0,5.666667,1.0,0.0,6.554444,0.0
…,…,…,…,…,…,…,…,…,…,…,…
"""Lodging chat""",2026-04-25 07:00:00,2026-04-24 17:00:00,14.5,0.5,0.0,0.0,5.0,0.0,0.0,0.0,0.0
"""Non-Lodging chat""",2026-04-11 22:00:00,2026-04-11 08:00:00,3.5,0.5,0.0,0.0,2.0,0.0,0.5,0.0,0.0
"""Lodging chat""",2026-02-23 02:00:00,2026-02-22 11:00:00,12.5,0.0,7.5,0.0,0.0,10.0,0.0,4.507778,0.608333


In [68]:
# --- IC HCM DETAILS ---
# Filter specifically for HCM and join with FINAL_AGGREGATION (Scheduled + Actual)
IC_HCM_Details_Log = Final_Merged_Data.filter(
    pl.col("Site") == "Concentrix (Ho Chi Minh City)"
).join(
    FINAL_AGGREGATION,
    on=["LOB", "VNT_Intervals", "PST_Intervals"],
    how="left"
).select([
    "LOB", "PST_Month", "PST_Date", "PST_Intervals", "PST_Interval_Range",
    "VNT_Date", "VNT_Intervals", "VNT_Interval_Range",
    
    "Forecast Productive Hour (Sum)", "Forecast Heads", 
    "Total_LOB_Productive_Hours", "Actual Heads",
    "Staffing Attainment (Pct)", "Interval Compliance (Pct)", "IC Status", "Occupancy", "Billable Hours (Sum)",
    
    "Site", "Req_Site_Level", "Site Req Heads", 
    "Site_Productive_Hours", "Site Actual Heads",
    "Staffing Attainment (Pct)_Site_Level", "Occupancy_Site_Level", "Handle Count (Sum)",
    
    "Scheduled_Open_Time", "Scheduled_Break", "Scheduled_Lunch", 
    "Scheduled_Training/Coaching", "Scheduled_Leave", "Scheduled_NCNS",
    
    "Actual_Break", "Actual_Lunch", "Actual_Training_Coaching"
])

In [69]:
# --- IC FORECASTING ---
forecast_unpl_pct = 0.10

# 1. Base query from OU Data
ou_forecast_base = OU_Data.filter(pl.col("Site") == "Concentrix (Ho Chi Minh City)").select([
    "LOB", "Site", "PST_Intervals", "VNT_Intervals", "Req"
]).rename({"Req": "Req_Site_Level"})

# 2. Join with IEX & Execute mathematical conversions
ic_forecast = ou_forecast_base.join(
    IEX_Aggregated, on=["LOB", "VNT_Intervals", "PST_Intervals"], how="left"
).with_columns([
    # Convert Hours to Heads (Multiply by 2)
    (pl.col("Req_Site_Level") * 2).alias("Site Req Heads"),
    (pl.col("Scheduled_Open_Time") * 2).alias("Scheduled_Open_Time"),
    (pl.col("Scheduled_Break") * 2).alias("Scheduled_Break"),
    (pl.col("Scheduled_Lunch") * 2).alias("Scheduled_Lunch"),
    (pl.col("Scheduled_Training/Coaching") * 2).alias("Scheduled_Training/Coaching"),
    (pl.col("Scheduled_Leave") * 2).alias("Scheduled_Leave"),
    (pl.col("Scheduled_NCNS") * 2).alias("Scheduled_NCNS"),
    
    # Base Coverage %
    pl.when(pl.col("Req_Site_Level") == 0).then(None).otherwise(
        (pl.col("Scheduled_Open_Time") * 2) / (pl.col("Req_Site_Level") * 2)
    ).alias("Forecast_Coverage_Pct"),
    
    # Date/Time Formatting
    pl.col("PST_Intervals").dt.strftime("%Y-%m").alias("PST_Month"),
    pl.col("PST_Intervals").dt.truncate("1w").dt.date().alias("PST_Week"),
    pl.col("PST_Intervals").dt.date().alias("PST_Date"),
    pl.concat_str([
        pl.col("PST_Intervals").dt.strftime("%H:%M"), pl.lit("-"),
        (pl.col("PST_Intervals") + pl.duration(minutes=30)).dt.strftime("%H:%M")
    ]).alias("PST_Interval_Range"),
    
    pl.col("VNT_Intervals").dt.date().alias("VNT_Date"),
    pl.concat_str([
        pl.col("VNT_Intervals").dt.strftime("%H:%M"), pl.lit("-"),
        (pl.col("VNT_Intervals") + pl.duration(minutes=30)).dt.strftime("%H:%M")
    ]).alias("VNT_Interval_Range")
]).with_columns([
    # Summing unproductive categories
    (
        pl.col("Scheduled_Break").fill_null(0) + pl.col("Scheduled_Lunch").fill_null(0) +
        pl.col("Scheduled_Training/Coaching").fill_null(0) + pl.col("Scheduled_Leave").fill_null(0) +
        pl.col("Scheduled_NCNS").fill_null(0)
    ).alias("Scheduled_Unproductive"),
    
    # Shrinkage mapping & HC loss calculation (Round up to nearest 0.5)
    pl.lit(forecast_unpl_pct).alias("Forecast_Unpl_Pct"),
    (((pl.col("Scheduled_Open_Time").fill_null(0) * forecast_unpl_pct) * 2).ceil() / 2).alias("Forecast_Lost_HC")
]).with_columns([
    # Adjusted coverages using HC loss
    pl.when(pl.col("Site Req Heads").fill_null(0) == 0).then(None).otherwise(
        (pl.col("Scheduled_Open_Time").fill_null(0) - pl.col("Forecast_Lost_HC")) / pl.col("Site Req Heads")
    ).alias("Forecast_2_Coverage_Pct"),
    
    (((pl.col("Scheduled_Open_Time").fill_null(0) - pl.col("Forecast_Lost_HC") - pl.col("Site Req Heads").fill_null(0)) * 2).ceil() / 2).alias("Forecast_HC_Surplus_Shortage")
]).select([
    "LOB", "PST_Month", "PST_Week", "PST_Date", "PST_Intervals", "PST_Interval_Range",
    "VNT_Date", "VNT_Intervals", "VNT_Interval_Range", "Site", "Req_Site_Level", "Site Req Heads",
    "Scheduled_Open_Time", "Scheduled_Break", "Scheduled_Lunch", "Scheduled_Training/Coaching",
    "Scheduled_Leave", "Scheduled_NCNS", "Scheduled_Unproductive", "Forecast_Coverage_Pct",
    "Forecast_Unpl_Pct", "Forecast_Lost_HC", "Forecast_HC_Surplus_Shortage", "Forecast_2_Coverage_Pct"
])

ic_forecast

LOB,PST_Month,PST_Week,PST_Date,PST_Intervals,PST_Interval_Range,VNT_Date,VNT_Intervals,VNT_Interval_Range,Site,Req_Site_Level,Site Req Heads,Scheduled_Open_Time,Scheduled_Break,Scheduled_Lunch,Scheduled_Training/Coaching,Scheduled_Leave,Scheduled_NCNS,Scheduled_Unproductive,Forecast_Coverage_Pct,Forecast_Unpl_Pct,Forecast_Lost_HC,Forecast_HC_Surplus_Shortage,Forecast_2_Coverage_Pct
str,str,date,date,datetime[μs],str,date,datetime[μs],str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Lodging chat""","""2025-12""",2025-12-29,2025-12-29,2025-12-29 00:00:00,"""00:00-00:30""",2025-12-29,2025-12-29 15:00:00,"""15:00-15:30""","""Concentrix (Ho Chi Minh City)""",2.455,4.91,null,null,null,null,null,null,0.0,null,0.1,0.0,-4.5,0.0
"""Lodging chat""","""2025-12""",2025-12-29,2025-12-29,2025-12-29 00:30:00,"""00:30-01:00""",2025-12-29,2025-12-29 15:30:00,"""15:30-16:00""","""Concentrix (Ho Chi Minh City)""",2.425,4.85,null,null,null,null,null,null,0.0,null,0.1,0.0,-4.5,0.0
"""Lodging chat""","""2025-12""",2025-12-29,2025-12-29,2025-12-29 01:00:00,"""01:00-01:30""",2025-12-29,2025-12-29 16:00:00,"""16:00-16:30""","""Concentrix (Ho Chi Minh City)""",2.05,4.1,null,null,null,null,null,null,0.0,null,0.1,0.0,-4.0,0.0
"""Lodging chat""","""2025-12""",2025-12-29,2025-12-29,2025-12-29 01:30:00,"""01:30-02:00""",2025-12-29,2025-12-29 16:30:00,"""16:30-17:00""","""Concentrix (Ho Chi Minh City)""",2.255,4.51,null,null,null,null,null,null,0.0,null,0.1,0.0,-4.5,0.0
"""Lodging chat""","""2025-12""",2025-12-29,2025-12-29,2025-12-29 02:00:00,"""02:00-02:30""",2025-12-29,2025-12-29 17:00:00,"""17:00-17:30""","""Concentrix (Ho Chi Minh City)""",1.925,3.85,null,null,null,null,null,null,0.0,null,0.1,0.0,-3.5,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Non-Lodging chat""","""2026-04""",2026-04-13,2026-04-19,2026-04-19 21:30:00,"""21:30-22:00""",2026-04-20,2026-04-20 11:30:00,"""11:30-12:00""","""Concentrix (Ho Chi Minh City)""",1.855,3.71,10.0,0.0,0.0,0.0,2.0,0.0,2.0,2.695418,0.1,1.0,5.5,2.425876
"""Non-Lodging chat""","""2026-04""",2026-04-13,2026-04-19,2026-04-19 22:00:00,"""22:00-22:30""",2026-04-20,2026-04-20 12:00:00,"""12:00-12:30""","""Concentrix (Ho Chi Minh City)""",1.38,2.76,4.0,2.0,4.0,0.0,2.0,0.0,8.0,1.449275,0.1,0.5,1.0,1.268116
"""Non-Lodging chat""","""2026-04""",2026-04-13,2026-04-19,2026-04-19 22:30:00,"""22:30-23:00""",2026-04-20,2026-04-20 12:30:00,"""12:30-13:00""","""Concentrix (Ho Chi Minh City)""",1.845,3.69,6.0,0.0,4.0,0.0,2.0,0.0,6.0,1.626016,0.1,1.0,1.5,1.355014


In [70]:
# --- DATA EXPORT ---
dt_format = "%Y-%m-%d %H:%M:%S"

# Dictionary holding filename and its corresponding dataframe
exports = {
    "Agent_Avoidance_RawData.csv": Agent_Avoidance_RawData,
    "Schedule_Adherence_Gap.csv": Schedule_Adherence_Gap,
    "Agent_Rest_Compliance_Log.csv": Agent_Rest_Compliance_Log,
    "IC_Details_Log.csv": Final_Merged_Data,
    "IC_HCM_Details_Log.csv": IC_HCM_Details_Log,
    "IC_Forecast.csv": ic_forecast
}

# Loop through and write each CSV cleanly
for filename, df in exports.items():
    df.write_csv(
        os.path.join(output_folder, filename),
        datetime_format=dt_format,
        include_bom=True
    )
    
print(f"Successfully exported {len(exports)} files to {output_folder}")

Successfully exported 6 files to C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_NON_COMPLIANCE
